In [ ]:
import tensorflow as tf
import numpy as np
import sklearn as sk
import os

In [ ]:
data = np.loadtxt('stock_sma_14_50_curated.csv', delimiter=',')

data = data.T
np.random.shuffle(data)
data = data.T

print(data.shape)

In [ ]:
X = data[:504,:]
Y = data[-1:,:]

print(X.shape)
print(Y.shape)

print(np.mean(Y))
print(np.std(Y))

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-8), input_shape=(504,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-8)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-8)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='relu'),
])

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.Huber(delta=0.5),
    metrics=['mae']
)

model.summary()


In [ ]:
# Clip extreme outliers in Y so MSE isn't dominated by rare spikes
# (p99 ≈ 2.09, but max ≈ 19 — those few outliers monopolize the gradient)
y_lo = np.percentile(Y, 2)
y_hi = np.percentile(Y, 98)
print(f"Clipping Y to [{y_lo:.4f}, {y_hi:.4f}]  (was [{Y.min():.4f}, {Y.max():.4f}])")
Y = np.clip(Y, y_lo, y_hi)

In [ ]:
train_ratio = 0.8
train_size = int(X.shape[1] * train_ratio)

X_train = X[:,:train_size].T
Y_train = Y[:,:train_size].T

X_val = X[:,train_size:].T
Y_val = Y[:,train_size:].T

print(X_train.shape)
print(X_val.shape)

In [ ]:
# Normalize features
#mean = X_train.mean(axis=0)
#std  = X_train.std(axis=0) + 1e-8
#X_train = (X_train - mean) / std
#X_val   = (X_val   - mean) / std

# Normalize Y — without this the output layer fights a non-zero offset
# and the model collapses to predicting the mean/median
y_mean = Y_train.mean()
y_std  = Y_train.std() + 1e-8
Y_train_norm = (Y_train - y_mean) / y_std
Y_val_norm   = (Y_val   - y_mean) / y_std
print(f"Y normalized: mean→0, std→1  (y_mean={y_mean:.4f}, y_std={y_std:.4f})")


In [ ]:
model.fit(X_train, Y_train,
                  validation_data=(X_val, Y_val),
                  batch_size=64, 
                  epochs=20,
                  verbose=1,
                  shuffle=True)


In [ ]:
import matplotlib.pyplot as plt

FIGURE_FOLDER = 'figures/'

def plot_acc_and_loss(history, title: str, save: bool = False):
    file_title = title.replace(' ', '-').lower()

    if save and not os.path.exists(FIGURE_FOLDER): os.mkdir(FIGURE_FOLDER)

    plt.title("Model and Validation MAE for " + title)
    xs = list(range(1, len(history.history['mae']) + 1))
    plt.plot(xs, history.history['mae'], label="Model MAE", color="Red")
    plt.plot(xs, history.history['val_mae'], label="Validation MAE", color="Blue")
    plt.xlabel("Epoch")
    plt.ylabel("MAE")
    plt.legend()

    if save: 
        plt.savefig(FIGURE_FOLDER + file_title + '-mae.png')
        plt.close()
    else: plt.show()

    plt.title("Model and Validation Loss for " + title)
    xs = list(range(1, len(history.history['val_loss']) + 1))
    plt.plot(xs, history.history['loss'], label="Model loss", color="Red")
    plt.plot(xs, history.history['val_loss'], label="Validation loss", color="Blue")
    plt.xlabel("Epoch")
    plt.ylabel("Cost")
    plt.legend()
    
    if save: 
        plt.savefig(FIGURE_FOLDER + file_title + '-loss.png')
        plt.close()
    else: plt.show()

def prcnt_within_tolerance(X, Y, tolerance):
    return np.count_nonzero(np.abs(X - Y) <= tolerance) / X.shape[0]

def r2_score(model, X, Y):
    return sk.metrics.r2_score(Y, model.predict(X))

# Difference Formula for real numbers
def mean_prcnt_diff(model, X, Y):
    abs_Y = np.abs(Y)
    abs_Pred = np.abs(model.predict(X))

    raw_diffs = (np.abs(abs_Pred - abs_Y) / ((abs_Pred + abs_Y) / 2))

    #print(raw_diffs[:10])

    return np.mean(raw_diffs[np.isfinite(raw_diffs)])

In [ ]:
plot_acc_and_loss(model.history, "Model MAE")

In [ ]:
print("Prediction:", model.predict(X_val[:10]) * y_std + y_mean)
print("Label:", Y_val[:10])

In [ ]:
modelMSE = tf.keras.Sequential([
    tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(504,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.BatchNormalization(),
    #tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.Dense(1, activation='linear'),
])

modelMSE.compile(
    optimizer='adam',
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=['mae']
)

modelMSE.summary()

In [ ]:
modelMSE.fit(X_train, Y_train,
                  validation_data=(X_val, Y_val),
                  batch_size=64, 
                  epochs=30,
                  verbose=1,
                  shuffle=True)

In [ ]:
plot_acc_and_loss(modelMSE.history, "MSE Model")

In [ ]:
print("Prediction:", modelMSE.predict(X_val[:15]))
print("Label:", Y_val[:15])

In [ ]:
#print("Hubard Within 5%:", prcnt_within_tolerance(X_val, Y_val_norm, model, 0.05))
#print("Hubard Within 10%:", prcnt_within_tolerance(X_val, Y_val_norm, model, 0.1))
#print("Hubard Within 10%:", prcnt_within_tolerance(X_val, Y_val_norm, model, 0.2))

In [ ]:
y_std = Y.std()

print("MSE Within 0.125 std (Mean is 0.0987):", prcnt_within_tolerance(X_val, Y_val, modelMSE, y_std / 8))
print("MSE Within 0.25 std (Mean is 0.1974):", prcnt_within_tolerance(X_val, Y_val, modelMSE, y_std / 4))
print("MSE Within 0.5 std (Mean is 0.383):", prcnt_within_tolerance(X_val, Y_val, modelMSE, y_std / 2))
print("MSE Within 1 std (Mean is 0.68):", prcnt_within_tolerance(X_val, Y_val, modelMSE, y_std))
r2_score(modelMSE)

In [ ]:
modelLG = tf.keras.Sequential([
    tf.keras.layers.Dense(756, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(756,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(756, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(756, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(756, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(756, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(756, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.BatchNormalization(),
    #tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.Dense(1, activation='linear'),
])

modelLG.compile(
    optimizer='adam',
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=['mae']
)

modelLG.summary()

In [ ]:
modelLG.fit(X_train, Y_train_norm,
                  validation_data=(X_val, Y_val_norm),
                  batch_size=64, 
                  epochs=20,
                  verbose=1,
                  shuffle=True)

In [ ]:
modelWD = tf.keras.Sequential([
    tf.keras.layers.Dense(1000, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(252,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(500, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='linear'),
])

modelWD.compile(
    optimizer='adam',
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=['mae']
)

modelWD.summary()

In [ ]:
modelWD.fit(X_train, Y_train_norm,
                  validation_data=(X_val, Y_val_norm),
                  batch_size=64, 
                  epochs=20,
                  verbose=1,
                  shuffle=True)

In [ ]:
print("WD Within 0.125(Avg is 0.0987):", prcnt_within_tolerance(X_val, Y_val_norm, modelWD, 0.125))
print("WD Within 0.25 (Avg is 0.1974):", prcnt_within_tolerance(X_val, Y_val_norm, modelWD, 0.25))
print("WD Within 0.5 (Avg is 0.383):", prcnt_within_tolerance(X_val, Y_val_norm, modelWD, 0.5))
print("WD Within 1 (Avg is 0.68):", prcnt_within_tolerance(X_val, Y_val_norm, modelWD, 1))

In [ ]:
plot_acc_and_loss(modelWD.history, "WD Model")

In [ ]:
modelLSTM = tf.keras.Sequential([
    tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(252,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Reshape((252, 1)),
    tf.keras.layers.LSTM(252),
    tf.keras.layers.Dense(1, activation='linear'),
])

modelLSTM.compile(
    optimizer='adam',
    loss=tf.keras.losses.MeanSquaredError(),
    metrics=['mae']
)

modelLSTM.summary()

In [ ]:
modelLSTM.fit(X_train, Y_train_norm,
                  validation_data=(X_val, Y_val_norm),
                  batch_size=64, 
                  epochs=20,
                  verbose=1,
                  shuffle=True)

In [ ]:
print("LSTM Within 0.125(Avg is 0.0987):", prcnt_within_tolerance(X_val, Y_val_norm, modelLSTM, 0.125))
print("LSTM Within 0.25 (Avg is 0.1974):", prcnt_within_tolerance(X_val, Y_val_norm, modelLSTM, 0.25))
print("LSTM Within 0.5 (Avg is 0.383):", prcnt_within_tolerance(X_val, Y_val_norm, modelLSTM, 0.5))
print("LSTM Within 1 (Avg is 0.68):", prcnt_within_tolerance(X_val, Y_val_norm, modelLSTM, 1))

In [ ]:
plot_acc_and_loss(modelLSTM.history, "LSTM Model")

In [ ]:
def mean_quadratic_error(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    squared_error = tf.square(y_pred - y_true)
    return tf.reduce_mean(squared_error, axis=-1)


modelMQE = tf.keras.Sequential([
    tf.keras.layers.Dense(253, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(253,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(253, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(253, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(126, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
    tf.keras.layers.Dense(1, activation='linear'),
])

modelMQE.compile(
    optimizer='adam',
    loss=mean_quadratic_error,
    metrics=['mae']
)

modelMQE.summary()

In [ ]:
historyMQE = modelMQE.fit(
    X_train,
    Y_train_norm,
    validation_data=(X_val, Y_val_norm),
    batch_size=64,
    epochs=20,
    verbose=1,
    shuffle=True,
)

In [ ]:
plot_acc_and_loss(historyMQE, "Custom MQE Model")

print("Prediction:", modelMQE.predict(X_val[:15]) * y_std + y_mean)
print("Label:", Y_val[:15])

In [ ]:
RESULTS_FOLDER = 'results/'

def train_sma_mse(window: int, offset: int, normalized: bool = False, log: bool = False):
    file = f'stock_sma_{window}_{offset}_curated.csv'

    data = np.loadtxt(file, delimiter=',')

    data = data.T
    np.random.shuffle(data)
    data = data.T

    X = data[:504,:]
    Y = data[-1:,:]

    y_lo = np.percentile(Y, 2)
    y_hi = np.percentile(Y, 98)
    
    if log: print(f"Clipping Y to [{y_lo:.4f}, {y_hi:.4f}]  (was [{Y.min():.4f}, {Y.max():.4f}])")
    
    Y = np.clip(Y, y_lo, y_hi)

    train_ratio = 0.8
    train_size = int(X.shape[1] * train_ratio)

    X_train = X[:,:train_size].T
    Y_train = Y[:,:train_size].T

    X_val = X[:,train_size:].T
    Y_val = Y[:,train_size:].T

    if normalized:
        y_mean = Y_train.mean()
        y_std  = Y_train.std() + 1e-8
        Y_train = (Y_train - y_mean) / y_std
        Y_val   = (Y_val   - y_mean) / y_std

    modelMSE = tf.keras.Sequential([
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(504,)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.BatchNormalization(),
        #tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.Dense(1, activation='linear'),
    ])

    modelMSE.compile(
        optimizer='adam',
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=['mae']
    )

    if log: modelMSE.summary()

    modelMSE.fit(X_train, Y_train,
        validation_data=(X_val, Y_val),
        batch_size=64, 
        epochs=30,
        verbose=log,
        shuffle=True)
    
    plot_acc_and_loss(modelMSE.history, ("Normalized " if normalized else "") + f"MSE {window} {offset} Model", save=True)

    y_std = Y.std()
    y_mean = np.full(X_val.shape[0], Y.mean())
    y_mean = y_mean.reshape((X_val.shape[0], 1))

    val_preds = modelMSE.predict(X_val)

    very_close = prcnt_within_tolerance(val_preds, Y_val, y_std / 8) / prcnt_within_tolerance(y_mean, Y_val, y_std / 8)
    close = prcnt_within_tolerance(val_preds, Y_val, y_std / 4) / prcnt_within_tolerance(y_mean, Y_val, y_std / 4)
    moderate = prcnt_within_tolerance(val_preds, Y_val, y_std / 2) / prcnt_within_tolerance(y_mean, Y_val, y_std / 2)
    far = prcnt_within_tolerance(val_preds, Y_val, y_std) / prcnt_within_tolerance(y_mean, Y_val, y_std)
    val_r2 = r2_score(modelMSE, X_val, Y_val)
    total_r2 = r2_score(modelMSE, X.T, Y.T)
    prcnts_off = mean_prcnt_diff(modelMSE, X_val, Y_val)
    
    if not os.path.exists(RESULTS_FOLDER): os.mkdir(RESULTS_FOLDER)

    file_title = f'mse-{window}-{offset}-results.csv' if not normalized else f'normalized-mse-{window}-{offset}-results.csv'

    with open(RESULTS_FOLDER + file_title, 'w') as f:
        f.write('very_close,close,moderate,far,val_r2,total_r2,prcnts_off\n')
        f.write(f'{very_close},{close},{moderate},{far},{val_r2},{total_r2},{prcnts_off}\n')


In [ ]:
train_sma_mse(14, 14)

In [ ]:
train_sma_mse(3, 3, normalized=True)

In [ ]:
train_sma_mse(50, 50)

In [ ]:
train_sma_mse(50, 14)

In [ ]:
interval_list = [3,7,15,31,63,126]

for window in interval_list:
    for offset_len in interval_list:
        train_sma_mse(window, offset_len)
        print(f"Completed window:{window}, offset:{offset_len}")